# Pseudobulk model building — CLAMP

**Environment:** `clamp-analyses`

Builds CLAMPbase (unsupervised) and CLAMPfull (GO Biological Process prior) models for every pseudobulk dataset in `data/pseudobulk/`. Preprocesses raw counts from `bulk_expr.csv` using `cpmCLAMP → preprocessCLAMP → zscoreCLAMP`. Outputs written to `output/01_model_building/05_pseudobulk/<dataset>/CLAMPbase/` and `.../CLAMPfull/`.

## Libraries

In [1]:
library(data.table)
library(dplyr)
library(rsvd)
library(Matrix)
library(here)
library(CLAMP)
library(PCAtools)

set.seed(123)



Attaching package: ‘dplyr’




The following objects are masked from ‘package:data.table’:

    between, first, last




The following objects are masked from ‘package:stats’:

    filter, lag




The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



Loading required package: ggplot2



Loading required package: ggrepel




Attaching package: ‘PCAtools’




The following objects are masked from ‘package:stats’:

    biplot, screeplot




## Configuration

In [2]:
DATASET  = "PBMC_Perez2022"
MAX_ITER = 500L
OUT_ROOT = "output/01_model_building/05_pseudobulk"
DATA_DIR = "data/pseudobulk"

## Download BP pathway GMT (cached)

In [3]:
gmt_raw <- list(
  BP = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025")
)
for (lib in names(gmt_raw)) {
  names(gmt_raw[[lib]]) <- paste0(lib, "_", names(gmt_raw[[lib]]))
}
pathMat_raw <- gmtListToSparseMat(gmt_raw)
cat("Pathway matrix:", nrow(pathMat_raw), "genes x", ncol(pathMat_raw), "pathways\n")

Auto-detected name: GO_Biological_Process_2025



Using cached file for GO_Biological_Process_2025



Pathway matrix: 14674 genes x 5343 pathways


## Build models for each dataset

In [4]:
message("========== ", DATASET, " ==========")
out_dir <- file.path(here(), OUT_ROOT, DATASET)
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# Load preprocessed data
norm_dt    <- fread(file.path(here(), OUT_ROOT, DATASET, "norm.csv"))
norm_genes <- norm_dt[[1]]
norm       <- as.matrix(norm_dt[, -1, with = FALSE])
storage.mode(norm) <- "numeric"
rownames(norm) <- norm_genes
samples <- colnames(norm)
cat(DATASET, "norm:", nrow(norm), "genes x", ncol(norm), "samples\n")

# Load k
k <- as.integer(read.csv(file.path(here(), OUT_ROOT, DATASET, "k.csv"))$k[1])
message("  k = ", k)

# SVD (needed by CLAMPbase and CLAMPfull)
g_fb       <- nrow(norm)
samples_fb <- ncol(norm)
SVD_K      <- floor((min(g_fb, samples_fb) - 1) / 4)
svdres <- rsvd(norm, k = SVD_K)

# CLAMPbase
message("  Running CLAMPbase ...")
baseRes <- CLAMPbase(Y = norm, svdres = svdres, clamp_k = k, trace = FALSE)

baseRes$Z <- data.frame(baseRes$Z); rownames(baseRes$Z) <- norm_genes
baseRes$B <- data.frame(baseRes$B); colnames(baseRes$B) <- samples

base_dir <- file.path(out_dir, "CLAMPbase")
dir.create(base_dir, showWarnings = FALSE, recursive = TRUE)
write.csv(baseRes$B, file.path(base_dir, "B.csv"))
write.csv(baseRes$Z, file.path(base_dir, "Z.csv"))
saveRDS(baseRes, file.path(base_dir, "CLAMPbase.rds"))
message("  CLAMPbase saved -> ", base_dir)

# Match BP prior to dataset genes
matched <- getMatchedPathwayMat(pathMat_raw, norm_genes)
cat("  Prior matched:", nrow(matched), "genes x", ncol(matched), "pathways\n")

# CLAMPfull
message("  Running CLAMPfull ...")
fullRes <- CLAMPfull(
  Y                 = norm,
  svdres            = svdres,
  priorMat          = matched,
  clamp.base.result = baseRes,
  use_cpp           = TRUE,
  trace             = FALSE,
  max.iter          = MAX_ITER,
  clamp_k           = k
)

fullRes$Z <- data.frame(fullRes$Z); rownames(fullRes$Z) <- norm_genes
fullRes$B <- data.frame(fullRes$B); colnames(fullRes$B) <- samples

full_dir <- file.path(out_dir, "CLAMPfull")
dir.create(full_dir, showWarnings = FALSE, recursive = TRUE)
write.csv(fullRes$B, file.path(full_dir, "B.csv"))
write.csv(fullRes$Z, file.path(full_dir, "Z.csv"))
if (!is.null(fullRes$summary)) {
  sumdf <- fullRes$summary %>%
    dplyr::rename(LV = LV_index) %>%
    dplyr::mutate(LV = paste0("LV", LV))
  write.csv(sumdf, file.path(full_dir, "summary.csv"), row.names = FALSE)
}
saveRDS(fullRes, file.path(full_dir, "CLAMPfull.rds"))
message("  CLAMPfull saved -> ", full_dir)

========== Lung_Sikkema2023 ==========



Lung_Sikkema2023 norm: 17145 genes x 100 samples


  k = 18



  Running CLAMPbase ...



****



...

Updating Z for CV



crossValidation



There are 18 LVs with AUC>0.70



There are 11 LVs with AUC>0.90



  CLAMPfull saved -> /home/msubirana/Documents/pivlab/clamp-analyses/output/01_model_building/05_pseudobulk/Lung_Sikkema2023/CLAMPfull

